# argmax-accuracy-eval — worked example 3: Token-level accuracy ignoring a pad index

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `argmax-accuracy-eval`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For sequence models the logits are `(B, T, C)` and labels are `(B, T)`. `argmax(dim=-1)` still picks the class per token. To score fairly you must exclude padding positions: build a mask `labels != pad_idx` and average `correct` only over the valid tokens.

## Worked solution

**Step 1 — token predictions.** `preds = logits.argmax(dim=-1)` collapses the class axis, giving a `(B, T)` tensor of predicted token classes — same shape as `labels`.

**Step 2 — correctness grid.** `correct = (preds == labels)` is `(B, T)` bool. But some of those positions are padding and shouldn't count.

**Step 3 — valid-token mask.** `valid = (labels != pad_idx)` is `(B, T)` bool, True only on real tokens.

**Step 4 — masked mean.** We want `(correct AND valid).sum() / valid.sum()`. Numerator: `(correct & valid).sum()` counts correct *real* tokens. Denominator: `valid.sum()` counts real tokens. Casting to float and dividing gives accuracy over non-pad positions only.

**Why it works.** A plain `.mean()` would dilute the score with always-trivial pad positions (or count mispredicted pads as wrong). Masking restores the metric to 'fraction of genuine tokens classified correctly'.

In [ ]:
def masked_token_accuracy(logits, labels, pad_idx):
    preds = logits.argmax(dim=-1)            # (B, T)
    correct = (preds == labels)              # (B, T) bool
    valid = (labels != pad_idx)              # (B, T) bool
    num = (correct & valid).sum().float()
    den = valid.sum().float()
    return (num / den).item()

t.manual_seed(0)
B, T, C = 2, 6, 7
logits = t.randn(B, T, C)
labels = t.randint(0, C, (B, T))
labels[0, -2:] = 0          # mark some positions as pad (idx 0)
print(round(masked_token_accuracy(logits, labels, pad_idx=0), 4))